In [2]:
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms


# ============================================================
# 1. Model definitions
# Must match the model used during training
# ============================================================

class ResidualConvBlock(nn.Module):
    def __init__(
        self,
        in_ch,
        out_ch,
        kernel_size=3,
        stride=1,
        padding=1,
        dilation=1
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_ch,
            out_ch,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            padding_mode="reflect"
        )

        self.conv2 = nn.Conv2d(
            out_ch,
            out_ch,
            kernel_size=kernel_size,
            stride=1,
            padding=padding,
            dilation=dilation,
            padding_mode="reflect"
        )

        self.relu = nn.LeakyReLU(0.1, inplace=True)

        if in_ch != out_ch:
            self.shortcut = nn.Conv2d(in_ch, out_ch, kernel_size=1)
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.relu(out)
        out = self.conv2(out)

        out = out + identity
        out = self.relu(out)

        return out


class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1, skip_scale=0.7):
        super().__init__()

        self.skip_scale = skip_scale

        self.up = nn.ConvTranspose2d(
            in_ch,
            out_ch,
            kernel_size=2,
            stride=2
        )

        self.conv = ResidualConvBlock(
            in_ch=out_ch * 2,
            out_ch=out_ch,
            kernel_size=kernel_size,
            padding=padding
        )

    def forward(self, x1, x2):
        x1 = self.up(x1)

        diff_y = x2.size(2) - x1.size(2)
        diff_x = x2.size(3) - x1.size(3)

        x1 = F.pad(
            x1,
            [
                diff_x // 2,
                diff_x - diff_x // 2,
                diff_y // 2,
                diff_y - diff_y // 2
            ]
        )

        x = torch.cat([self.skip_scale * x2, x1], dim=1)
        return self.conv(x)


class ResUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, base_features=32):
        super().__init__()

        f = base_features

        self.inc = ResidualConvBlock(
            in_channels,
            f,
            kernel_size=5,
            padding=2
        )

        self.down1 = nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=2),
            ResidualConvBlock(f, f * 2)
        )

        self.down2 = nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=2),
            ResidualConvBlock(f * 2, f * 4)
        )

        self.down3 = nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=2),
            ResidualConvBlock(f * 4, f * 8)
        )

        self.up1 = UpBlock(f * 8, f * 4, skip_scale=0.7)
        self.up2 = UpBlock(f * 4, f * 2, skip_scale=0.7)
        self.up3 = UpBlock(f * 2, f, skip_scale=0.7)

        self.outc = nn.Conv2d(
            f,
            out_channels,
            kernel_size=3,
            padding=1,
            padding_mode="reflect"
        )

        self.res_scale = nn.Parameter(torch.tensor(0.2))

    def forward(self, x):
        decoded_input = x

        pad = 16
        x = F.pad(x, (pad, pad, pad, pad), mode="reflect")

        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)

        x = self.up1(x4, x3)
        x = self.up2(x, x2)
        x = self.up3(x, x1)

        residual = self.outc(x)
        residual = residual[..., pad:-pad, pad:-pad]

        enhanced = decoded_input + self.res_scale * residual

        if self.training:
            return enhanced

        return torch.clamp(enhanced, 0.0, 1.0)


# ============================================================
# 2. Utility functions
# ============================================================

def load_model(checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)

    grayscale = checkpoint.get("grayscale", False)
    base_features = checkpoint.get("base_features", 32)

    in_channels = 1 if grayscale else 3
    out_channels = in_channels

    model = ResUNet(
        in_channels=in_channels,
        out_channels=out_channels,
        base_features=base_features
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    print(f"Loaded checkpoint from epoch {checkpoint.get('epoch', 'unknown')}")
    print(f"Validation loss: {checkpoint.get('val_loss', 'unknown')}")
    print(f"Base features: {base_features}")
    print(f"Grayscale: {grayscale}")

    return model, grayscale


def tensor_to_pil(tensor):
    """
    tensor: [1, C, H, W] in [0, 1]
    """
    tensor = tensor.squeeze(0).detach().cpu().clamp(0.0, 1.0)

    array = tensor.numpy()

    if array.shape[0] == 1:
        array = array[0]
        array = array * 255.0
        return Image.fromarray(array.astype(np.uint8), mode="L")

    array = np.transpose(array, (1, 2, 0))
    array = array * 255.0

    return Image.fromarray(array.astype(np.uint8), mode="RGB")


# ============================================================
# 3. Full-image inference
# ============================================================

@torch.no_grad()
def enhance_full_image(model, image_tensor, device):
    """
    Use this if the full image fits in GPU memory.

    image_tensor: [1, C, H, W]
    """
    image_tensor = image_tensor.to(device)
    enhanced = model(image_tensor)
    return enhanced


# ============================================================
# 4. Tiled inference for large images
# ============================================================

@torch.no_grad()
def enhance_image_tiled(
    model,
    image_tensor,
    device,
    tile_size=512,
    overlap=64
):
    """
    Tiled inference with averaging in overlapping areas.

    image_tensor: [1, C, H, W]

    tile_size should be divisible by 8 or 16.
    overlap helps avoid seams between tiles.
    """

    model.eval()

    image_tensor = image_tensor.to(device)

    _, C, H, W = image_tensor.shape

    output = torch.zeros_like(image_tensor, device=device)
    weight = torch.zeros_like(image_tensor, device=device)

    stride = tile_size - overlap

    y_positions = list(range(0, H, stride))
    x_positions = list(range(0, W, stride))

    for y in y_positions:
        for x in x_positions:
            y0 = y
            x0 = x
            y1 = min(y0 + tile_size, H)
            x1 = min(x0 + tile_size, W)

            # Move tile window back if we are near the border
            y0 = max(0, y1 - tile_size)
            x0 = max(0, x1 - tile_size)

            tile = image_tensor[:, :, y0:y1, x0:x1]

            enhanced_tile = model(tile)

            output[:, :, y0:y1, x0:x1] += enhanced_tile
            weight[:, :, y0:y1, x0:x1] += 1.0

    output = output / weight.clamp_min(1e-8)
    output = torch.clamp(output, 0.0, 1.0)

    return output


# ============================================================
# 5. Main
# ============================================================

def main():
    checkpoint_path = "models/best_resunet_jpeg_artifact_removal.pth"

    input_image_path = "../src/Dataset/processed/0049.png"
    output_image_path = "../src/Dataset/outputs/0049_enhanced.png"

    use_tiling = True

    tile_size = 512
    overlap = 64

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model, grayscale = load_model(checkpoint_path, device)

    image = Image.open(input_image_path)

    if grayscale:
        image = image.convert("L")
    else:
        image = image.convert("RGB")

    to_tensor = transforms.ToTensor()

    image_tensor = to_tensor(image).unsqueeze(0)

    print(f"Input tensor shape: {image_tensor.shape}")

    if use_tiling:
        enhanced_tensor = enhance_image_tiled(
            model=model,
            image_tensor=image_tensor,
            device=device,
            tile_size=tile_size,
            overlap=overlap
        )
    else:
        enhanced_tensor = enhance_full_image(
            model=model,
            image_tensor=image_tensor,
            device=device
        )

    enhanced_image = tensor_to_pil(enhanced_tensor)

    output_image_path = Path(output_image_path)
    output_image_path.parent.mkdir(parents=True, exist_ok=True)

    enhanced_image.save(output_image_path)

    print(f"Saved enhanced image to: {output_image_path}")


if __name__ == "__main__":
    main()

Using device: cpu
Loaded checkpoint from epoch 39
Validation loss: 0.030967558175325392
Base features: 32
Grayscale: False
Input tensor shape: torch.Size([1, 3, 1344, 2032])
Saved enhanced image to: ../src/Dataset/outputs/0049_enhanced.png
